# 2. Data Preprocessing - Akkadian to English Translation
## Aaron Dichoso & Luis Razon

This notebook details the steps performed for preprocessing the cleaned dataset used in the Deep Past Challenge for Translating Akkadian Text to English.
The competition can be accessed in this link: https://www.kaggle.com/competitions/deep-past-initiative-machine-translation/data

Run this notebook AFTER running "1. Cleaning".

In [1]:
import pandas as pd
import numpy
print(numpy.__version__)
print(pd.__version__)

cleaned_complete_df = pd.read_csv("processed/cleaned_train_complete.csv")
cleaned_incomplete_df = pd.read_csv("processed/cleaned_train_incomplete.csv")

cleaned_complete_df.sample(5)

2.4.1
3.0.0


,oare_id,transliteration,translation
1352,dfa4f36e-eb92-45fd-800c-514ba0af5966,<big_gap> i-dì-in-ma ša-lim-a-šùr ù DUMU a-mur...,"<big_gap> the colony rendered a verdict, and Š..."
880,8e6f8e72-25e0-44fe-a498-4818636a8861,a-na a-lá-hi-im qí-bi-ma um-ma <big_gap> šu-ma...,To Ali-ahum from <big_gap>: If you observe <bi...
18,02df45ee-6d74-46fb-9870-3e546ce7d304,a-na PUZUR4-(d)IM lá-dí-a en-um-a-šur en-na-sú...,"To Puzur-Adad, Ladiya, Ennam-Aššur, Enna-Suen,..."
1487,f74875b8-6f58-4bfa-a4da-ac136a0e406a,um-ma ša-lim-a-šur-ma a-na i-dí-a-bi-im e-na-n...,"From Šalim-Aššur to Iddin-abum, Ennānum, Inah-..."
968,9b14060a-4325-4e83-aab8-2e717d743480,kà-ru-um ṣa-he-er GAL dí-nam i-dí-in-ma ša-lim...,The plenary assembly of the colony passed a ve...


The goal of this notebook is to tokenize the data using Byte-Pair Encoding so that it can be trained on networks that can accept sequential data like LSTM and transformers.
First, we look for all text in the transliteration that are contained in parentheses to find the determinatives. These are prefixes/suffixes attached to words that give it special meanings.

In [2]:
#Parentheses in the cleaned data were not removed for determinatives, so look inside these for them.
def parse(text):
    stack = []
    for char in text:
        if char == '(':
            #stack push
            stack.append([])
        elif char == ')':
            yield ''.join(stack.pop())
        elif len(stack) > 0:
            #stack peek
            stack[-1].append(char)
    
    return stack

parses = []
for i, row in pd.concat([cleaned_complete_df, cleaned_incomplete_df]).iterrows():
    parses.append(tuple(parse(row['transliteration'])))

#Loop through parse tuples
determinatives = dict()
for p in parses:
    for d in p:
        if determinatives.get(d):
            determinatives[d] += 1
        else:
            determinatives[d] = 1

determinatives

{'d': 482, 'ki': 383, 'TÚG': 192, 'HI': 1}

These determinatives are isolated and then tokenized into corresponding english meanings according to the dataset. 

In [3]:
DET_TOKENS = {
    "(d)": "<god>",
    "(HI)": "<star>",
    "(ki)": "<place>",
    "(lu2)": "<person>",
    "(e2)": "<building>",
    "(uru)": "<city>",
    "(kur)": "<land>",
    "(mi)": "<female>",
    "(m)": "<male>",
    "(gesh)": "<wood>",
    "(TÚG)": "<textile>",
    "(dub)": "<tablet>",
    "(id2)": "<river>",
    "(mushen)": "<bird>",
    "(na4)": "<stone>",
    "(kush)": "<hide>",
    "(u2)": "<plant>",
}

In [4]:
for key in DET_TOKENS.keys():
    cleaned_complete_df['transliteration'] = cleaned_complete_df['transliteration'].str.replace(key, DET_TOKENS[key])
    cleaned_incomplete_df['transliteration'] = cleaned_incomplete_df['transliteration'].str.replace(key, DET_TOKENS[key])

In [5]:
#Sanity Checkpoint: continually repeat running this cell to view your processed data so far.
cleaned_complete_df.sample(5)

,oare_id,transliteration,translation
432,4433bc7d-6c00-4d83-a763-c6a6921c3537,1/2 ma-na KÙ.BABBAR ṣa-ru-pá-am iš-tí a-šùr-na...,1/2 mina of refined silver due from Aššur-nādā...
1051,aa84dc4a-b81b-4519-9d60-797b7ba276c8,KIŠIB kà-ri-im kà-ni-iš kà-ru-um dí-nam i-dí-i...,Seal of the Kanesh colony. The colony has pass...
78,0bc255d7-4dcb-4763-8d8b-2d4af41720b4,2 GÚ 10 ma-na AN.NA ku-nu-ki 52 TÚG SIG5 3 ANŠ...,"2 talents 10 minas of sealed tin, 52 good text..."
276,2c0fa8ce-2871-4912-a4c9-f01f07f56dc0,a-na be-lu-ba-ni ù a-lá-hi-im qí-bi-ma um-ma z...,To Bēlum-bāni and Ali-ahum from Zukuwa: 2 extr...
48,07e7a1b1-4d8d-4363-996b-e29d43fe77e4,1 ku-ta-num 2 šu-re-en6 ša šu-ku-bi-im a-na 2/...,1 textile and 2 dark textiles belonging to Šu-...


In [ ]:
import re

#Additional cleaning done because of tokenization. Separate all tokens that are attached to words with spaces to prep the dataset for BPE.
#NOTE: Try NOT running this and see if model results can change. This is a completely optional preprocessing step.
cleaned_complete_df['transliteration'] = cleaned_complete_df['transliteration'].str.replace(re.compile(r'(\w+)*<(\w+)>(\w+)*'), r'\1 <\2> \3', regex=True)
cleaned_incomplete_df['transliteration'] = cleaned_incomplete_df['transliteration'].str.replace(re.compile(r'(\w+)*<(\w+)>(\w+)*'), r'\1 <\2> \3', regex=True)

In [ ]:
#Sanity Checkpoint.
cleaned_complete_df.sample(5)

,oare_id,transliteration,translation
869,8c167b5b-efde-466d-a6d2-06faf49a637c,i-na 1-ma-na KÙ.BABBAR ša a-na a-lim <place> ...,From the 1 mina of silver which I had Bēlānum ...
602,5f264cf3-196b-4105-9d75-b6a6aedaae97,šu-ma um-me-a-an ša-lim-a-šur a-na AN.NA ù <b...,If Šalim-Ašsur's investors raise claim against...
734,772ba716-bbe5-4bbf-949f-ac4767147189,KIŠIB a-lu-lá-a-na en-nam-a-šur,Sealed by Alulaya to Ennam-Aššur.
1203,c3cbcb45-835e-4139-a6e2-ced1d7882bca,<big_gap> -na GÍR ša a-šur iṣ-ba-at-ma a-šur-...,<big_gap>na seized Aššur's dagger and Aššur-ma...
996,a1251a33-431a-401a-b678-d8b20181ddda,4 me-at sí-pá-ra-tum 18 šé-na-tum 34 šé-na-tum...,"400 bronze pins, 18 shoes, 34 children's shoes..."


In [ ]:
cleaned_complete_df.to_csv("processed/processed_train_complete_untokenized.csv")
cleaned_incomplete_df.to_csv("processed/processed_train_incomplete_untokenized.csv")

## Tokenized Dataset

To tokenize the dataset, we first encode the characters into tokens. We use Byte-Pair Encoding (BPE) for this, as it offers a good balance between vocabulary size and character length compared to other encoding techniques like character encoding (small vocab size, but does not learn spelling) and word encoding (large vocabulary). 

Byte-Pair Encoding works by finding the most common substrings across the dataset and using those as individual tokens to be used for encoding (Source: https://www.geeksforgeeks.org/nlp/byte-pair-encoding-bpe-in-nlp/). Thus, it lends itself well to the Akkadian transliterations, which are made up of several substrings seperated by dashes. the determinatives pre/appended on words are also a key characteristic in the Akkadian transliterations which BPE naturally fits for this use case.

In [8]:
import utils.bpe as bpe
import importlib

importlib.reload(bpe)

BPE = bpe.BytePairEncoder()
BPE_akk = bpe.BytePairEncoder()
cleaned_complete_df["transliteration"] = cleaned_complete_df["transliteration"].astype("object")
cleaned_incomplete_df["transliteration"] = cleaned_incomplete_df["transliteration"].astype("object")
cleaned_complete_df["translation"] = cleaned_complete_df["translation"].astype("object")
cleaned_incomplete_df["translation"] = cleaned_incomplete_df["translation"].astype("object")

#Get the whole text from across the dataset for BPE
akkadian_only = ""
whole_text = ""
for i, row in pd.concat([cleaned_complete_df, cleaned_incomplete_df]).iterrows():
    #Seperate entries by <sos> (Start of Sequence) and <eos> (End of Sequence) tokens.
    new_row = " <sos> " + row['transliteration'] + " <eos> "
    akkadian_only += new_row
    new_row += " <sos> " + row['translation'] + " <eos> "
    whole_text += new_row

#Fit BPE onto the text.
BPE.fit(whole_text, 4000)
BPE_akk.fit(akkadian_only, 1000)

for k in BPE.vocab.keys():
    print(k, ":", BPE.vocab[k])

#Save the vocabulary and tokens
BPE.save("processed/akk2eng.json")
BPE_akk.save("processed/akkonly.json")

<sos> : 3122
KIŠIB_ : 520
ma_ : 13214
-_ : 129103
nu_ : 2825
ba_ : 2031
lúm_ : 214
a_ : 21141
šur_ : 2063
DUMU_ : 1901
ṣí_ : 609
lá_ : 4466
<god>IM_ : 75
šu_ : 4625
<god>EN.LÍL_ : 79
ki_ : 1638
MAN_ : 24
ta_ : 2937
1/3_ : 750
na_ : 10982
2_ : 1529
GÍN_ : 1860
KÙ.BABBAR_ : 3388
SIG5_ : 324
i_ : 5986
ṣé_ : 445
er_ : 777
PUZUR4_ : 240
hu_ : 993
um_ : 3379
iš_ : 1764
tù_ : 1415
ha_ : 1657
muš_ : 44
tim_ : 1869
ša_ : 6178
ì_ : 269
lí_ : 425
dan_ : 184
ITU.KAM_ : 200
ke_ : 82
li_ : 2367
mu_ : 871
e_ : 1855
sú_ : 1029
in_ : 2534
I TU_ : 4
14_ : 99
am_ : 1898
qal_ : 179
qú_ : 535
ul_ : 435
1/2_ : 1603
GÍN.TA_ : 185
1_ : 1965
im_ : 2730
ITU.1.KAM_ : 79
ib_ : 432
tám_ : 625
ú_ : 3632
ṣa_ : 748
áb_ : 437
<eos> : 3122
Seal_ : 162
of_ : 8815
Mannum_ : 93
balum_ : 28
Aššur_ : 2290
son_ : 1921
Ṣilli_ : 10
Adad,_ : 28
seal_ : 488
Šu_ : 813
Illil_ : 52
kī_ : 73
Aššur,_ : 341
Puzur_ : 360
Ataya._ : 8
Ataya_ : 20
owes_ : 208
22_ : 80
shekels_ : 1520
good_ : 203
silver_ : 2646
to_ : 3951
Ali_ : 599
ahum._

### Encode Text into tokens
WARNING. THIS CELL TAKES VERY LONG TO EXECUTE (~11 Hours). Be warned.

NOTE: The same operations in this notebook is copied onto compiled/preprocess.py. This can be more easily executed in a cloud server if you have one.

In [ ]:
for i, row in cleaned_complete_df.iterrows():
    new_row = " <sos> " + row['transliteration'] + " <eos> "
    encoded = BPE.encode(new_row)
    cleaned_complete_df.at[i, 'transliteration'] = encoded

    new_row = " <sos> " + row['translation'] + " <eos> "
    encoded = BPE.encode(new_row)
    cleaned_complete_df.at[i, 'translation'] = encoded

for i, row in cleaned_incomplete_df.iterrows():
    new_row = " <sos> " + row['transliteration'] + " <eos> "
    encoded = BPE.encode(new_row)
    cleaned_incomplete_df.at[i, 'transliteration'] = encoded

    new_row = " <sos> " + row['translation'] + " <eos> "
    encoded = BPE.encode(new_row)
    cleaned_incomplete_df.at[i, 'translation'] = encoded

['<sos>', 'KIŠIB_', 'ma', '-', 'nu', '-', 'b', 'a', '-', 'l', 'ú', 'm', '-', 'a', '-', 'šur_', 'DUMU_', 'ṣ', 'í', '-', 'lá', '-_', '<god>', 'IM_', 'KIŠIB_', 'š', 'u', '-_', '<god>', 'EN.LÍL_', 'DUMU_', 'ma', '-', 'nu', '-', 'ki', '-', 'a', '-', 'šur_', 'KIŠIB_', 'M', 'AN', '-', 'a', '-', 'šur_', 'DUMU_', 'a', '-', 'ta', '-', 'a_', '1/3_', 'ma', '-', 'na_', '2_', 'GÍN_', 'KÙ.BABBAR_', 'SIG5_', 'i', '-', 'ṣ', 'é', '-', 'er_', 'PUZUR', '4', '-', 'a', '-', 'šur_', 'DUMU_', 'a', '-', 'ta', '-', 'a', '-', 'lá', '-', 'hu', '-', 'um_', 'i', '-', 'šu_', 'iš', '-', 'tù_', 'ha', '-', 'mu', 'š', '-', 'tim_', 'ša_', 'ì', '-', 'l', 'í', '-', 'dan_', 'ITU.KAM_', 'ša_', 'ke', '-', 'na', '-', 'tim_', 'li', '-', 'mu', '-', 'um_', 'e', '-', 'na', '-', 's', 'ú', '-', 'in_', 'a', '-', 'na_', 'I', 'TU_', '14_', 'ha', '-', 'am', '-', 'š', 'a', '-', 'tim_', 'i', '-', 'š', 'a', '-', 'qal_', 'š', 'u', '-', 'ma_', 'lá_', 'iš', '-', 'q', 'ú', '-', 'ul_', '1/2_', 'GÍN.TA_', 'a', '-', 'na_', '1_', 'ma', '-', 'na', 

In [ ]:
#Sanity Checkpoint. All rows should be an array of tokens at this point.
cleaned_complete_df.sample(5)

,oare_id,transliteration,translation
1208,c4509217-0fed-4328-8bc6-fb9fb41f229e,"[<sos>, 15_, GÍN_, KÙ.BABBAR_, i, -, na_, li, ...",15 shekels of silver is owed by Irišum son of ...
700,71112263-5009-4065-b5ff-d4ec32dd83bb,"[<sos>, a, -, na_, e, -, lá, -, ma_, q, í, -, ...","Say to Elamma, thus Aššur-rē'ī: The 2 Šulupkae..."
120,121f01b4-5a21-4703-8eee-425d568648d7,"[<sos>, um, -, ma_, ku, -, z, i, -, z, i, -, a...",From Kuziziya to Ennam-Aššur: Šu-Bēlum and Alu...
813,82b2c169-af37-4c2b-bfdd-0fa305d42fe0,"[<sos>, š, a, -, li, m, -, a, -, šùr_, ù_, š, ...",Šalim-Aššur and Šu-Bēlum son of Iddin-abum set...
353,37ea781c-687b-4e96-b8a5-c100ddc27f30,"[<sos>, um, -, ma_, a, -, šù, r, -, š, a, -, d...","Thus say Aššur-šad-ilī and Ababaya, say to Kul..."


In [ ]:
cleaned_complete_df.to_csv("processed/processed_train_complete.csv")
cleaned_incomplete_df.to_csv("processed/processed_train_incomplete.csv")

Data Preprocessing Ends Here.